In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
def list_files_in_subdirs(parent_dir: str | Path, subdir_names: list[str]) -> list[Path]:
    """
    Given a parent directory and a list of subdirectory names,
    return a list of Path objects for all files inside each subdirectory
    (non-recursive — only direct contents of each subdir).
    """
    parent_dir = Path(parent_dir)
    all_files = []

    for name in subdir_names:
        subdir = parent_dir / name
        if not subdir.is_dir():
            print(f"Warning: {subdir} is not a valid directory, skipping.")
            continue
        all_files.extend(p for p in subdir.iterdir() if p.is_file())

    return all_files


In [6]:
import statistics
from pathlib import Path
import librosa

def audio_length_stats(file_paths: list[str | Path]) -> tuple[float, float]:
    """
    Given a list of audio file paths, return (mean, stdev) of their durations in seconds.
    Uses librosa to compute duration.
    """
    durations = []

    for path in file_paths:
        path = Path(path)
        try:
            duration = librosa.get_duration(path=str(path))
            durations.append(duration)
        except Exception as e:
            print(f"Warning: error reading {path}: {e}, skipping.")

    if not durations:
        raise ValueError("No valid audio files found to compute stats.")

    mean_len = statistics.mean(durations)
    stdev_len = statistics.stdev(durations) if len(durations) > 1 else 0.0

    return mean_len, stdev_len

In [4]:
sub_dir_lst = ["Explosion", "Gunshots", "UAV"]
parent_path = Path(r"..\Data\Audio_Files\Raw_Audios\Anomalies")

audio_files = list_files_in_subdirs(parent_path, sub_dir_lst)

In [7]:
mean, stdev = audio_length_stats(audio_files)

In [ ]:
print(mean)
print(stdev) #in seconds

18.281575907479393
34.67026782856111


In [10]:
import numpy as np
import librosa
import soundfile as sf
from pathlib import Path

def shorten_audio_files(input_dir: str | Path, seg_mean: float = 18.0, seg_stdev: float = 34.0, min_len: float = 10.0):
    """
    For every audio file in input_dir, sample a segment length from
    N(seg_mean, seg_stdev), extract a random segment of that length,
    and save it as .wav in a sibling directory "shortened_unlabeled".
    """
    input_dir = Path(input_dir)
    output_dir = input_dir.parent / "shortened_unlabeled"
    output_dir.mkdir(exist_ok=True)

    audio_extensions = {".wav", ".mp3", ".flac", ".ogg", ".m4a", ".aac"}

    for file_path in input_dir.iterdir():
        if not file_path.is_file() or file_path.suffix.lower() not in audio_extensions:
            continue

        try:
            full_duration = librosa.get_duration(path=str(file_path))

            # Sample segment length, take abs value, floor at min_len
            seg_len = abs(np.random.normal(seg_mean, seg_stdev))
            seg_len = max(seg_len, min_len)
            seg_len = min(seg_len, full_duration)  # can't exceed file's own length

            max_start = full_duration - seg_len
            start = np.random.uniform(0, max_start) if max_start > 0 else 0.0

            y, sr = librosa.load(str(file_path), sr=None, offset=start, duration=seg_len)

            out_path = output_dir / (file_path.stem + ".wav")
            sf.write(str(out_path), y, sr)

            print(f"{file_path.name}: {seg_len:.2f}s segment @ {start:.2f}s -> {out_path.name}")

        except Exception as e:
            print(f"Error processing {file_path}: {e}")

    print(f"\nDone. Files saved to: {output_dir}")

In [15]:
f_path = Path(r".\Data_v2\Raw_Audios\Test\Unlabeled")
shorten_audio_files(f_path)

658728__performansas__lithuanian-voice-acting.wav: 17.02s segment @ 0.00s -> 658728__performansas__lithuanian-voice-acting.wav
665234__drdock__approving-voice-male.wav: 3.66s segment @ 0.00s -> 665234__drdock__approving-voice-male.wav
665940__joseviana__amanhecer-na-ilha-com-cachorros-galos-e-barcos-sunrise-on-the-island-with-rooster-dogs-and-boats.wav: 20.38s segment @ 370.65s -> 665940__joseviana__amanhecer-na-ilha-com-cachorros-galos-e-barcos-sunrise-on-the-island-with-rooster-dogs-and-boats.wav
675423__craigsmith__s15-16-horse-trots-in-stable-movement-in-background.wav: 38.19s segment @ 0.20s -> 675423__craigsmith__s15-16-horse-trots-in-stable-movement-in-background.wav
688235__philip_goddard__river-teign-at-night-from-sharp-tor-teign-gorge.flac: 10.77s segment @ 2526.25s -> 688235__philip_goddard__river-teign-at-night-from-sharp-tor-teign-gorge.wav
690720__kevp888__voc_230611-2268_fr_thunderstorm.wav: 10.00s segment @ 791.16s -> 690720__kevp888__voc_230611-2268_fr_thunderstorm.wav

In [ ]:
""" 
Next step is to bootstrap data for final files that will be used for
training testing

Need alot of data for CNN model!
"""

In [3]:
from Models import AudioBootstrapper
from pathlib import Path

bootstrapper = AudioBootstrapper()

og_folder_path = Path(r".\Data_v2\Raw_Audios\Test")
sub_folder_names = ["Explosion", "Gunshots", "shortened_unlabeled", "UAV"] 
output_folder_path = Path(r".\Data_v2\Bootstrapped_Audios\Test") 
overlay_folder_name = "shortened_unlabeled"
aug_technique_num = 4 
num_bootstraps = 75

bootstrapper.BootStrap(og_folder_path, sub_folder_names, output_folder_path, overlay_folder_name, aug_technique_num, num_bootstraps)

2026-07-02 11:53:24,583 - INFO - Files processed: 1/300
2026-07-02 11:53:24,750 - INFO - Files processed: 2/300
2026-07-02 11:53:24,953 - INFO - Files processed: 3/300
2026-07-02 11:53:24,990 - INFO - Files processed: 4/300
2026-07-02 11:53:25,144 - INFO - Files processed: 5/300
2026-07-02 11:53:25,283 - INFO - Files processed: 6/300
2026-07-02 11:53:25,311 - INFO - Files processed: 7/300
2026-07-02 11:53:25,371 - INFO - Files processed: 8/300
2026-07-02 11:53:25,477 - INFO - Files processed: 9/300
2026-07-02 11:53:25,676 - INFO - Files processed: 10/300
2026-07-02 11:53:25,701 - INFO - Files processed: 11/300
2026-07-02 11:53:25,758 - INFO - Files processed: 12/300
2026-07-02 11:53:25,808 - INFO - Files processed: 13/300
2026-07-02 11:53:25,872 - INFO - Files processed: 14/300
2026-07-02 11:53:25,936 - INFO - Files processed: 15/300
2026-07-02 11:53:26,061 - INFO - Files processed: 16/300
2026-07-02 11:53:26,437 - INFO - Files processed: 17/300
2026-07-02 11:53:26,464 - INFO - Files p

{'file_name': ['audio_1.wav',
  'audio_2.wav',
  'audio_3.wav',
  'audio_4.wav',
  'audio_5.wav',
  'audio_6.wav',
  'audio_7.wav',
  'audio_8.wav',
  'audio_9.wav',
  'audio_10.wav',
  'audio_11.wav',
  'audio_12.wav',
  'audio_13.wav',
  'audio_14.wav',
  'audio_15.wav',
  'audio_16.wav',
  'audio_17.wav',
  'audio_18.wav',
  'audio_19.wav',
  'audio_20.wav',
  'audio_21.wav',
  'audio_22.wav',
  'audio_23.wav',
  'audio_24.wav',
  'audio_25.wav',
  'audio_26.wav',
  'audio_27.wav',
  'audio_28.wav',
  'audio_29.wav',
  'audio_30.wav',
  'audio_31.wav',
  'audio_32.wav',
  'audio_33.wav',
  'audio_34.wav',
  'audio_35.wav',
  'audio_36.wav',
  'audio_37.wav',
  'audio_38.wav',
  'audio_39.wav',
  'audio_40.wav',
  'audio_41.wav',
  'audio_42.wav',
  'audio_43.wav',
  'audio_44.wav',
  'audio_45.wav',
  'audio_46.wav',
  'audio_47.wav',
  'audio_48.wav',
  'audio_49.wav',
  'audio_50.wav',
  'audio_51.wav',
  'audio_52.wav',
  'audio_53.wav',
  'audio_54.wav',
  'audio_55.wav',
  'aud

In [ ]:
""" 
Note to prevent data leakage source audio were seperated into train test at the start
bootstrapped train test audios were constructed from seperate source files
"""